# 01 — Load the four sources into one monthly table

**Reads:** `data/raw/core_cpi_lseg_first_release_poll.csv`, `manheim_used_vehicle_value_index.xlsx`,
`fred_jet_fuel_monthly.csv`, `fred_cpi_used_cars.csv`.

**Writes:** `data/processed/monthly.csv`; `results/tables/data_coverage.csv`.

| Source | What it is |
|---|---|
| LSEG, *United States Core CPI MM, SA* | the monthly change in core CPI (all items less food and energy, seasonally adjusted) **as first published**, its release date, and the median of the Reuters poll of forecasters taken before the release |
| Cox Automotive, Manheim Used Vehicle Value Index | wholesale used-car prices, seasonally adjusted (Jan 1997 = 100) |
| FRED `DJFUELUSGULF` | Gulf Coast jet fuel, dollars per gallon, monthly average of daily prices |
| FRED `CUSR0000SETA02` | the CPI's own used-car index; used only in notebook 02 to check how far Manheim leads it |

The LSEG table could not be exported (the Excel add-in is disabled and the Python API needs a separate
licence), so it was captured from the Indicator Explorer table and typed into a CSV; see
`data/raw/fetch_log.json`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
for d in (PROC, TAB, FIG):
    d.mkdir(parents=True, exist_ok=True)

import openpyxl

REFETCH = False   # True downloads the two FRED files again (needs internet). LSEG and Manheim are manual downloads.
FRED_URLS = {
    "fred_jet_fuel_monthly.csv":
        "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DJFUELUSGULF&cosd=1999-01-01&fq=Monthly&fam=avg",
    "fred_cpi_used_cars.csv":
        "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CUSR0000SETA02&cosd=1997-01-01",
}
if REFETCH:
    import urllib.request
    for name, url in FRED_URLS.items():
        urllib.request.urlretrieve(url, RAW / name)
        print("downloaded", name)

## Core CPI: first prints, release dates and the Reuters poll

*Period* is the month the prices refer to; *release date* is the day the BLS published it (usually 10–15
days after the month ends). The first print is rounded to 0.1 percentage point, like the poll.

In [2]:
core = pd.read_csv(RAW / "core_cpi_lseg_first_release_poll.csv")
core["period"] = pd.PeriodIndex(core["period"], freq="M")
core["release_date"] = pd.to_datetime(core["release_datetime_lseg"]).dt.normalize()
core = (core.set_index("period")
            .rename(columns={"actual_first_release_pct": "core_first", "reuters_poll_median_pct": "poll"})
            [["release_date", "core_first", "poll"]])

full = pd.period_range(core.index.min(), core.index.max(), freq="M")
print("months in the LSEG table:", len(core), f"({core.index.min()} to {core.index.max()})")
print("months missing from the table:", [str(p) for p in full.difference(core.index)])
print("first prints start:", core["core_first"].first_valid_index())
print("poll blank in:", [str(p) for p in core.index[core["poll"].isna()]])
month_end = core.index.to_timestamp(how="end").normalize()
lag = (core["release_date"] - month_end).dt.days
print("days from the end of the month to the release:", int(lag.min()), "to", int(lag.max()))

months in the LSEG table: 326 (1999-06 to 2026-10)
months missing from the table: ['1999-12', '2025-10', '2025-11']
first prints start: 2004-04
poll blank in: ['2005-04', '2005-05', '2005-07', '2005-11', '2007-11']
days from the end of the month to the release: 10 to 30


October 2025 was never published (the BLS did not collect October prices during the government shutdown),
and November 2025 was published only as a two-month change, so LSEG has no one-month print for either.
Both months drop out as targets; notebook 02 builds the inputs from the prints that do exist.

## Manheim wholesale used-car index

In [3]:
ws = openpyxl.load_workbook(RAW / "manheim_used_vehicle_value_index.xlsx", data_only=True).worksheets[0]
rows = list(ws.iter_rows(values_only=True))
labels = [(r[0] or "").strip() for r in rows]
months = pd.PeriodIndex(pd.to_datetime(list(rows[0][1:]), format="%m-%Y"), freq="M")
manheim = pd.Series(rows[labels.index("Wholesale Used Vehicle Price Index")][1:], index=months,
                    dtype=float, name="manheim_index")
print(manheim.index.min(), "to", manheim.index.max(), "-", int(manheim.notna().sum()), "months")

1997-01 to 2026-10 - 358 months


## FRED: jet fuel and the CPI used-car index

In [4]:
def read_fred(name, column):
    s = pd.read_csv(RAW / name, parse_dates=["observation_date"]).set_index("observation_date")[column]
    s.index = s.index.to_period("M")
    return s

jet = read_fred("fred_jet_fuel_monthly.csv", "DJFUELUSGULF").rename("jet_fuel_usd_gal")
used = read_fred("fred_cpi_used_cars.csv", "CUSR0000SETA02").rename("cpi_used_cars_index")
print("jet fuel:", jet.first_valid_index(), "to", jet.last_valid_index())
print("CPI used cars:", used.first_valid_index(), "to", used.last_valid_index())

jet fuel: 1999-02 to 2026-10
CPI used cars: 1997-01 to 2026-10


## One monthly table

Month-on-month changes are in percent. A change is missing when either month is missing.

In [5]:
idx = pd.period_range("1997-01", "2026-10", freq="M")
monthly = pd.concat([core, manheim, jet, used], axis=1).reindex(idx)
monthly.index.name = "period"


def pct_change(s):
    return (s / s.shift(1) - 1) * 100


monthly["manheim_mom"] = pct_change(monthly["manheim_index"])
monthly["jet_mom"] = pct_change(monthly["jet_fuel_usd_gal"])
monthly["used_cars_mom"] = pct_change(monthly["cpi_used_cars_index"])

out = monthly.copy()
out.index = out.index.astype(str)
out.to_csv(PROC / "monthly.csv", float_format="%.6f")

cols = ["core_first", "poll", "manheim_index", "jet_fuel_usd_gal", "cpi_used_cars_index"]
coverage = pd.DataFrame({c: {"first month": str(monthly[c].first_valid_index()),
                             "last month": str(monthly[c].last_valid_index()),
                             "months with data": int(monthly[c].notna().sum())} for c in cols}).T
coverage.to_csv(TAB / "data_coverage.csv")
coverage

,first month,last month,months with data
core_first,2004-04,2026-10,269
poll,1999-06,2026-10,321
manheim_index,1997-01,2026-10,358
jet_fuel_usd_gal,1999-02,2026-10,333
cpi_used_cars_index,1997-01,2026-10,358
